In [37]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()

True

In [38]:
class JokeState(TypedDict):
    joke: str
    topic: str
    explanation: str

In [39]:
model = ChatGroq(model='llama-3.3-70b-versatile')

In [40]:
def create_joke(state: JokeState) -> dict:
    joke = model.invoke(f"Tell me a joke about {state['topic']}")
    return {"joke": joke.content}

In [41]:
def explain_joke(state: JokeState) -> dict:
    explanation = model.invoke(f"Explain the joke: {state['joke']}")
    return {"explanation": explanation.content}

In [42]:
graph = StateGraph(JokeState)

graph.add_node('create_joke', create_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'create_joke')
graph.add_edge('create_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpoint = InMemorySaver()                                #Adding persistence to the workflow by creating a checkpoint saver

joke_workflow = graph.compile(checkpointer=checkpoint)

### Create thread for workflow

In [46]:
config1 = {"configurable": {"thread_id": '1'}}

joke_workflow.invoke({'topic': 'myself'},config=config1)

{'joke': 'I don\'t know much about you, but I can try to come up with a lighthearted joke. Here goes:\n\n"You\'re so unique, you\'re the only person I\'ve met today who\'s talking to a language model... mainly because I\'m a large language model, I don\'t actually \'meet\' people in person. But still, that\'s pretty cool, right?"\n\nIf you\'d like a more personalized joke, feel free to share a bit about yourself, and I\'ll try to come up with something more tailored to your interests or sense of humor!',
 'topic': 'myself',
 'explanation': 'A meta joke. The joke is self-aware and pokes fun at the fact that it\'s being generated by a large language model. Here\'s how it works:\n\n1. The setup starts by saying "You\'re so unique," which is a common phrase used to compliment someone. However, the punchline subverts this expectation by revealing that the "uniqueness" is actually a result of the fact that the language model doesn\'t "meet" people in person.\n2. The humor comes from the fact

### Get Final State of the workflow thread

In [47]:
joke_workflow.get_state(config1)

StateSnapshot(values={'joke': 'I don\'t know much about you, but I can try to come up with a lighthearted joke. Here goes:\n\n"You\'re so unique, you\'re the only person I\'ve met today who\'s talking to a language model... mainly because I\'m a large language model, I don\'t actually \'meet\' people in person. But still, that\'s pretty cool, right?"\n\nIf you\'d like a more personalized joke, feel free to share a bit about yourself, and I\'ll try to come up with something more tailored to your interests or sense of humor!', 'topic': 'myself', 'explanation': 'A meta joke. The joke is self-aware and pokes fun at the fact that it\'s being generated by a large language model. Here\'s how it works:\n\n1. The setup starts by saying "You\'re so unique," which is a common phrase used to compliment someone. However, the punchline subverts this expectation by revealing that the "uniqueness" is actually a result of the fact that the language model doesn\'t "meet" people in person.\n2. The humor 

### Get History of the workflow thread

In [48]:
list(joke_workflow.get_state_history(config1))

[StateSnapshot(values={'joke': 'I don\'t know much about you, but I can try to come up with a lighthearted joke. Here goes:\n\n"You\'re so unique, you\'re the only person I\'ve met today who\'s talking to a language model... mainly because I\'m a large language model, I don\'t actually \'meet\' people in person. But still, that\'s pretty cool, right?"\n\nIf you\'d like a more personalized joke, feel free to share a bit about yourself, and I\'ll try to come up with something more tailored to your interests or sense of humor!', 'topic': 'myself', 'explanation': 'A meta joke. The joke is self-aware and pokes fun at the fact that it\'s being generated by a large language model. Here\'s how it works:\n\n1. The setup starts by saying "You\'re so unique," which is a common phrase used to compliment someone. However, the punchline subverts this expectation by revealing that the "uniqueness" is actually a result of the fact that the language model doesn\'t "meet" people in person.\n2. The humor